# AI-Generated Image Detection Using Deep Learning

**Institution**: Sreenidhi Institute of Science and Technology (Sreenidhi University)  
**Department**: Department of Computer Science and Engineering  
**Project Repository**: [GitHub Repository](https://github.com/LalithaSreya/DL-Mini-Project-AI-Image-Detection)  
**Dataset**: [CIFAKE: Real and AI-Generated Synthetic Images](https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images)  

---

## 1. Project Overview & Objectives
The rapid rise of generative artificial intelligence (such as Latent Diffusion Models and Generative Adversarial Networks) enables the synthesis of photorealistic images that are virtually indistinguishable from real photographs by the naked eye. This project develops a deep learning image classifier using **Transfer Learning with MobileNetV2** to distinguish between:
- **FAKE (0)**: AI-generated synthetic images (produced via Stable Diffusion v1.4)
- **REAL (1)**: Authentic natural photographs (from CIFAR-10)

### Google Colab Instructions for Beginners:
1. In the top menu, go to **Runtime** > **Change runtime type**, and select **T4 GPU** for accelerated training.
2. Execute each code cell sequentially by clicking the **Play (▶)** button on the left or pressing `Shift + Enter`.

## 2. Environment Setup & Dependency Installation
Install the required libraries and verify GPU availability.

In [ ]:
# Install dependencies if not already present
!pip install -q tensorflow numpy matplotlib scikit-learn pillow

import os
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("Available GPUs:", gpus if gpus else "Running on CPU (Tip: Switch to GPU runtime for faster training!)")

## 3. Google Drive Mounting & Dataset Acquisition
You can store the extracted dataset on your Google Drive or upload/download it directly into the Colab session.

In [ ]:
# Mount Google Drive (Optional, uncomment if your dataset is stored on Drive)
# from google.colab import drive
# drive.mount('/content/drive')

# OPTION B: Download directly from Kaggle in Colab (Requires your kaggle.json)
# If you have your kaggle.json on your computer, upload it to Colab, then uncomment:
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images -p /content/
# !unzip -q /content/cifake-real-and-ai-generated-synthetic-images.zip -d /content/CIFAKE

## 4. Dataset Path Configuration & Verification
> **EDITABLE CELL**: Set `DATA_DIR` below to the exact folder where the extracted CIFAKE dataset resides.

In [ ]:
# =======================================================================
# EDIT THIS PATH TO POINT TO YOUR DATASET DIRECTORY
# =======================================================================
DATA_DIR = Path("/content/CIFAKE")  # or "/content/drive/MyDrive/CIFAKE" or "data/CIFAKE"

# Verify folder structure
train_dir = DATA_DIR / "train"
test_dir = DATA_DIR / "test"

if not DATA_DIR.exists():
    print(f"[!] Directory '{DATA_DIR}' does not exist yet. Please download and extract the dataset first.")
else:
    print(f"[OK] Found dataset root at: {DATA_DIR.resolve()}")
    for split, p in [("train", train_dir), ("test", test_dir)]:
        if p.exists():
            fake_cnt = len(list((p / 'FAKE').glob('*.jpg')))
            real_cnt = len(list((p / 'REAL').glob('*.jpg')))
            print(f"  {split.upper()} split: FAKE={fake_cnt:,} images, REAL={real_cnt:,} images (Total={fake_cnt+real_cnt:,})")
        else:
            print(f"  [!] Missing split directory: {p}")

## 5. Hyperparameter Configuration & Dataset Loading
We use an input image size of 160x160x3 and batch size of 32.
- **Class 0**: `FAKE`
- **Class 1**: `REAL`

In [ ]:
IMAGE_SIZE = (160, 160)
BATCH_SIZE = 32
SEED = 42
CLASS_NAMES = ["FAKE", "REAL"]

if DATA_DIR.exists() and train_dir.exists() and test_dir.exists():
    print("Loading training split (80% train / 20% validation)...")
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        labels="inferred",
        label_mode="int",
        class_names=CLASS_NAMES,
        validation_split=0.2,
        subset="training",
        seed=SEED,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        labels="inferred",
        label_mode="int",
        class_names=CLASS_NAMES,
        validation_split=0.2,
        subset="validation",
        seed=SEED,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        test_dir,
        labels="inferred",
        label_mode="int",
        class_names=CLASS_NAMES,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
    test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)
    print("Datasets successfully initialized!")
else:
    print("Dataset directories not found. Please verify DATA_DIR before running loading.")

## 6. Model Architecture (MobileNetV2 Transfer Learning)
We instantiate a pretrained MobileNetV2 feature extractor trained on ImageNet:
1. **Input**: Raw RGB image `(160, 160, 3)`
2. **Preprocessing**: Internal `mobilenet_v2.preprocess_input` (rescales `[0, 255]` to `[-1, 1]`)
3. **Backbone**: MobileNetV2 with frozen ImageNet weights
4. **Pooling & Regularization**: `GlobalAveragePooling2D` + `Dropout(0.2)`
5. **Output**: Single sigmoid unit for binary probability `P(REAL)`

In [ ]:
def create_model(input_shape=(160, 160, 3), learning_rate=1e-4):
    inputs = tf.keras.layers.Input(shape=input_shape, name="input_image")
    x = tf.keras.layers.Rescaling(
        scale=1.0 / 127.5,
        offset=-1.0,
        name="mobilenetv2_preprocess",
    )(inputs)

    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights="imagenet",
    )
    base_model.trainable = False  # Freeze pretrained backbone

    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = tf.keras.layers.Dropout(0.2, name="dropout")(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="output_sigmoid")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="mobilenetv2_cifake")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
        metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy")],
    )
    return model

model = create_model()
model.summary()

## 7. Model Training
We train using `EarlyStopping` and `ModelCheckpoint` callbacks to preserve the optimal weights.

In [ ]:
EPOCHS = 5
checkpoint_path = "mobilenetv2_cifake_best.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
]

if 'train_ds' in locals():
    print("Starting model training...")
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
    )
else:
    print("train_ds not loaded. Please prepare the dataset before running training.")

## 8. Training History Visualization
Plot training and validation loss and accuracy curves across epochs.

In [ ]:
if 'history' in locals():
    epochs_range = range(1, len(history.history['loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Loss curve
    ax1.plot(epochs_range, history.history['loss'], 'b-o', label='Training Loss')
    ax1.plot(epochs_range, history.history['val_loss'], 'r--s', label='Validation Loss')
    ax1.set_title('Loss vs. Epochs')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Binary Cross-Entropy Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.5)

    # Accuracy curve
    ax2.plot(epochs_range, history.history['accuracy'], 'b-o', label='Training Accuracy')
    ax2.plot(epochs_range, history.history['val_accuracy'], 'g--^', label='Validation Accuracy')
    ax2.set_title('Accuracy vs. Epochs')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.5)

    plt.tight_layout()
    plt.show()
else:
    print("History unavailable because training has not been executed yet.")

## 9. Evaluation on Held-Out Test Split
Calculate test accuracy, loss, per-class classification report, and confusion matrix.

In [ ]:
if 'test_ds' in locals() and os.path.exists(checkpoint_path):
    eval_model = tf.keras.models.load_model(checkpoint_path)
    print("Evaluating best model on held-out test split...")
    test_loss, test_acc = eval_model.evaluate(test_ds, verbose=1)
    print(f"\nTest Loss: {test_loss:.4f} | Test Accuracy: {test_acc*100:.2f}%")

    y_true, y_pred = [], []
    for images, labels in test_ds:
        preds = eval_model.predict(images, verbose=0).flatten()
        y_pred.extend((preds >= 0.5).astype(int))
        y_true.extend(labels.numpy().flatten())

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, cmap=plt.cm.Blues)
    plt.title('Test Confusion Matrix')
    plt.colorbar()
    plt.xticks([0, 1], CLASS_NAMES)
    plt.yticks([0, 1], CLASS_NAMES)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    for i in range(2):
        for j in range(2):
            plt.text(j, i, f"{cm[i, j]:,}", ha='center', va='center',
                     color='white' if cm[i, j] > cm.max()/2 else 'black')
    plt.tight_layout()
    plt.show()
else:
    print("Evaluation pending: Trained checkpoint or test split not yet available.")

## 10. Single-Image Prediction Demonstration
Test an individual image with the detector and interpret its confidence score.

In [ ]:
def predict_single_image(image_path, model, target_size=(160, 160)):
    img = Image.open(image_path).convert("RGB")
    img_resized = img.resize(target_size, Image.Resampling.BILINEAR)
    img_array = np.expand_dims(np.array(img_resized, dtype=np.float32), axis=0)

    score = float(model.predict(img_array, verbose=0)[0][0])
    if score >= 0.5:
        pred_class = "REAL"
        confidence = score * 100.0
    else:
        pred_class = "FAKE"
        confidence = (1.0 - score) * 100.0

    plt.figure(figsize=(3, 3))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Prediction: {pred_class}\nConfidence: {confidence:.2f}%")
    plt.show()

    print(f"Predicted Class : {pred_class}")
    print(f"Model Confidence: {confidence:.2f}%")
    print(f"Raw Model Score : {score:.4f} (Sigmoid P(REAL))")

# Example invocation (replace with your test image path):
# if os.path.exists('sample.jpg') and 'eval_model' in locals():
#     predict_single_image('sample.jpg', eval_model)